In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-11: On-policy distillation: the mechanism

## RFT Alternative methods, part 2 of the RLLM series


## Where this session fits

> **Three problems are open: exposure bias (contested, RLLM-7), coarse credit (RLLM-8), DPO's stored states (RLLM-10). This session: one method that gives partial answers to all three.**

- **Part 2: RFT Alternative methods**: alternatives to the PPO and GRPO you know, each placed by three axes
- **Axes**: 1 whose samples (RLLM-7), 2 who supplies the weights (a reference or a reward), 3 how fine (RLLM-8)
- **Position**: session 11 of 14, after RLLM-10 (Preference pairs and DPO)
- **Goal of Part 2**: a **map** that places any method by the three axes

<img src="images/series_map_11.png" width=55%>


## This session: RLLM-11

- **Question**: what is **on-policy distillation** (OPD), what loss does it minimize at each position, and which KL order does it use?
- **By the end you can**:
  - say what OPD is: a **student** samples, a **teacher** (a frozen model giving next-token probabilities) reads its prefix, and the student moves toward it
  - say which of the **three earlier problems** it addresses, and which it only **replaces** (a closeness-to-teacher signal, not credit for outcomes)
  - write the loss as a **KL** (distance between two distributions): **forward** with the teacher's whole distribution, or a **reverse** estimate on one drawn token
  - say when forward KL **spreads** and reverse KL **commits**
  - place both versions on the map


## Three problems the earlier sessions left

> **Exposure bias, coarse credit, and DPO's stored states: each leaves the update short of something.**

| Problem | Left by | What a fix would need |
|---|---|---|
| **Exposure bias** (contested): training prefixes are the dataset's, generation prefixes the model's own | RLLM-7 | a signal at the **model's own** prefixes |
| **Coarse credit**: one number per completion | RLLM-8, axis 3 | a signal **per token** |
| **Stored states**: DPO compares whole completions written **earlier** | RLLM-10 | **fresh** states, and a finer signal |

- PPO with a critic (RLLM-8) already has all three: own prefixes, a number per token, fresh samples
- **On-policy distillation** gives a **dense, closeness-to-teacher signal at the student's own prefixes**
  - no learned critic, no reward function. It answers the three problems **only in part**


## Step 1 of 4: The idea: the student writes, the teacher reads

> **On-policy distillation (OPD): the student samples; the teacher reads the student's prefix and gives a distribution at every position; the student moves toward it.**

<img src="images/opd_loop.png" alt="sample from the student, teacher reads the same tokens, compare per position, update the student" width=52%>

<!-- seed-pay: teacher-forcing-prefix -->
- **Student**: the model being trained. **Teacher**: a frozen model whose next-token probabilities we read. Not necessarily stronger (RLLM-12)
- Sampling as in GRPO, with RLLM-7's **staleness dial**; no reward function
- From RLLM-7: **teacher forcing** trains on the **reference's** prefix. Here the teacher only **reads** the student's: not the same
- The teacher speaks at **states the student reaches**, mistakes included

> **Keep this: what does a teacher say at a state it would never reach? It returns in RLLM-12.**
<!-- seed-plant: unreached-state -->


## Step 2 of 4: KL divergence, in two lines

> **KL divergence measures how far $Q$ is from $P$. It is zero only when they agree, and the order of the two matters.**

$$\KL(P\,\Vert\,Q)=\sum_v P(v)\log\frac{P(v)}{Q(v)}\quad\text{(nats: natural log)}$$

*Why this section? The loop says "compare teacher and student". KL is the comparison, and it comes in two orders.*

- Two outcomes, $P=(0.9,\,0.1)$, $Q=(0.5,\,0.5)$: $\KL(P\Vert Q)=0.37$, but $\KL(Q\Vert P)=0.51$
- With $P_{\text{T}}$ the teacher's next-token distribution and $P_\theta$ the student's:
  - **forward KL** $=\KL(P_{\text{T}}\Vert P_\theta)$: weights each token by the **teacher**
  - **reverse KL** $=\KL(P_\theta\Vert P_{\text{T}})$: weights each token by the **student**
- RLLM-2 and RLLM-4's penalty is a reverse-order KL: $d=\log(\pi_\theta/\pi_{\text{ref}})$ over the policy's own tokens
  - the frozen **reference model** (anchor) sits where the teacher is here. The teacher is a target to approach, not a brake


## Step 2 of 4: One headline loss: a KL to the teacher at every position

> **At every position of the student's own completion, compare the teacher's distribution with the student's; average over positions. The teacher supplies a reference there.**

$$\loss=\frac1{\lvert\y\rvert}\sum_{\tt=1}^{\lvert\y\rvert}\ell_\tt,\qquad \ell_\tt=\text{a KL between }P_{\text{T}}(\cdot\mid\y_{<\tt})\text{ and }P_\theta(\cdot\mid\y_{<\tt})$$

*Why one number per position? We need one number per position to average.*

- $\y_{<\tt}$ is the student's prefix; gradients flow **only** through $P_\theta$
- **Two versions** of $\ell_\tt$: a design pairing, not a law (a full-vocabulary reverse KL is computable too)
  - **Full distribution, forward KL**: all of $P_{\text{T}}$, a weight $P_{\text{T}}(v)$ on **every** candidate $v$. A **reference** (axis 2)
  - **Sampled token, reverse-KL estimate**: one signed number on the **drawn** token. Next slide


## Step 2 of 4: The sampled-token version: one signed number from the teacher

> **The teacher signal $u_{\tt}=\log\pi_{\text{T}}(\y_{\tt})-\log\pi_\theta(\y_{\tt})$ is positive if the teacher likes the drawn token more than the student does.**

- It has the **form of a reward** (one signed number per token), but its source is the **teacher**, not a task outcome
- $u_{\tt}$ is a **stop-gradient constant**: the gradient is $u_{\tt}\,\nabla\log\pi_\theta(\y_{\tt})$
- RLLM-4's $d$ with the sign flipped, the teacher in the anchor's place
- **Why reverse KL**: the **student** draws the token, so the average is under the student. A forward estimate needs teacher-drawn tokens
- **What practitioners use** (Lu et al., Thinking Machines, in the reading)


## Step 2 of 4: The two versions at one position

> **Both versions can say "less". The full distribution does it through normalization; the sampled token does it through its sign.**

<img src="images/opd_reference_vs_reward.png" alt="full distribution versus sampled token, both from the teacher: the teacher probabilities as weights on every token, versus one signed number on the drawn token" width=62%>

- From RLLM-8: a push **down** needs a **negative** weight
- **Full distribution**: weights $P_{\text{T}}(v)\ge0$ sum to 1, so raising the teacher's favorites **lowers the rest**
  - cross-entropy with a **soft label**; SFT's is one-hot
- **Sampled token**: drawn **6** gives $u=\log\frac{0.20}{0.55}=-1.01$ (push **down**); drawn **5** gives $\log\frac{0.70}{0.30}=+0.85$ (up)
- Neither lacks "less". What OPD lacks is a **task reward**


## Step 2 of 4: Myopic caveat (sampled-token version)

> **Our derivation (the standard policy-gradient identity), not tested here: the sampled-token signal gives no credit for a token's effect on later positions.**

- It is an unbiased gradient of the **per-position** loss: a discount-0 policy gradient of the sequence objective
- A token that **sets up** a good or bad later position gets nothing for it
- So OPD **replaces** one credit per completion by **closeness to the teacher at every position**. It does not assign credit for outcomes


## Step 3 of 4: Which direction? One position, charged both ways

> **At one position both orders reach zero together; they differ in who pays for the mismatch.**

*Why this section? Step 2 gave two versions that differ by KL order. Does the order change what the student learns?*

| Next token | Teacher $P_{\text{T}}$ | Student $P_\theta$ | Forward term $P_{\text{T}}\log\frac{P_{\text{T}}}{P_\theta}$ | Reverse term $P_\theta\log\frac{P_\theta}{P_{\text{T}}}$ |
|---|---|---|---|---|
| 5 | 0.70 | 0.30 | +0.59 | -0.25 |
| 6 | 0.20 | 0.55 | -0.20 | +0.56 |
| 4 | 0.08 | 0.10 | -0.02 | +0.02 |
| other | 0.02 | 0.05 | -0.02 | +0.05 |
| **sum** | 1.00 | 1.00 | **0.35** | **0.37** |

- Synthetic: next token after the prefix "2 + 3 =". Forward weights by the teacher: most charge at **5**
  - reverse weights by the student: most charge at **6**
- The table shows the orders disagreeing about the **loss**, not the optimum: a student that matches the teacher reaches 0 in both


## Step 3 of 4: Forward spreads, reverse commits (an analogy)

> **An analogy, not a result: a student too limited to match a teacher with two good answers. Forward KL spreads over both, reverse KL commits to one.**

<img src="images/forward_reverse_kl.png" alt="forward KL fit of one bump to a two-bump teacher spreads over both; reverse KL commits to one" width=70%>

- If the student **cannot** match, which error is cheaper?
  - forward: **missing** teacher mass is costly
  - reverse: missing mass is **free**, **extra** student mass is costly
- A capacity effect needs **sequence level or shared parameters**: the one-bump student is **our reading**
- **Classical distillation**: a student trained on the teacher's **saved logits** at the **teacher's** prefixes. OPD's forward version uses the **student's** prefixes
- Reverse is the order of RLLM-2's KL penalty


## Step 3 of 4: Which order to use

- **A design pairing, not a law**: each version uses the order it makes cheap
  - the full-distribution version sums the whole vocabulary at each student prefix, so it is **not** an average over the student's tokens
- Questions that tilt the choice (arguments from the objectives, **not results**):
  - **Many acceptable answers, all wanted?** Forward keeps the teacher's spread
  - **One answer is right?** Committing is closer to what you want
- Which order matters in practice is **empirical**


## Step 4 of 4: Where OPD sits on the map

> **The RLLM-11 slot (own samples, a reference) is filled: a teacher reading the student's own prefix supplies the weights, in both versions.**

<img src="images/opd_map.png" alt="the map with the on-policy distillation slot filled and highlighted in the own-samples row, reference column, with the forward-KL and sampled-token versions; the hybrid box and DPO from earlier sessions in normal color" width=46%>

- **Both versions**: the **teacher** supplies the weights, a **reference**, on the student's own samples. No task reward
- **Placement rule**: axis 2 asks who supplies the weights, not their form
  - $u_{\tt}$ has the form of a reward, but the teacher supplies it: reference column (our reading)
- The **order** changes how many numbers the weights carry (axis 3), not who supplies them (axis 2)
- A placement says how a method is **built**, not whether it works (**our reading**)


## What it takes to run it

> **OPD removes the reward call. It does not remove the need for a teacher: how good, and at which states? That is RLLM-12.**

- **Same tokenizer** in both models, or the distributions are over different sets
- **Teacher probabilities** at every position of every sampled completion, not just text
- **A teacher forward pass per step**: no backward pass, but time and memory
- **Generation** is still inside the loop, as in GRPO
- Full-vocabulary losses store a $\lvert\y\rvert\times\lvert\Vocab\rvert$ table per model
- The dependency is **relocated**: GRPO needs a scorer every step. OPD needs a teacher **obtained once** and also **run every step**


## Think about

1. Method **W**: the student samples, the teacher writes its **own** completion, then SFT on that. Which map cell? What differs from OPD?
2. Compare $\ell_\tt$ where the teacher is **confident** and the student **uncertain** with where **both are uncertain**. Which loss is larger, by the table's pattern?
3. A task has **many acceptable answers**. **Predict** what forward and reverse KL do to the student's diversity. How to measure it?
4. The teacher is **no better** than the student at some prompts. What does OPD's loss do there? (RLLM-12)


## What we established, and what it sets up

- **Established**: in OPD a **teacher supplies the weights** (a reference) at every position, in two versions. It answers the three problems **in part**
- **Limit of this session**: it assumes a **teacher**, and says nothing about how good one must be, or at which states
- **Sets up**: RLLM-12, The teacher, and when it is not enough
  - a problem: **a teacher: how good, at which states?**


## Aside, justification: why forward KL penalizes missing teacher mass

*For your own review; not worked through in session.*

1. $\KL(P\Vert Q)=\sum_x P(x)\log\frac{P(x)}{Q(x)}$. *(Definition; $P$ the teacher, $Q$ the model.)*
2. Wherever $P(x)>0$ but $Q(x)\to0$, the term $\to\infty$. *(The log of a ratio with a vanishing denominator.)*
3. So a finite loss **requires** $Q$ to put mass wherever $P$ does. Concretely, $P=0.3$, $Q=0.001$ gives $0.3\log(0.3/0.001)\approx1.7$ nats from that one point.
4. For $\KL(Q\Vert P)=\sum_x Q(x)\log\frac{Q(x)}{P(x)}$ the sum is **weighted by $Q$**: a point with $Q(x)\approx0$ contributes $\approx0$ however large $P(x)$ is.

**In words:** forward KL **cannot afford to ignore** anything the teacher does; reverse KL **can**.


## Aside, justification: a compromise between the two orders (generalized JSD)

*For your own review; not worked through in session.*

1. Let $M_\beta=\beta P+(1-\beta)Q$ with $0<\beta<1$, a **mixture** of the teacher $P$ and the student $Q$.
2. Define $D_{\beta}(P,Q)=\beta\,\KL(P\Vert M_\beta)+(1-\beta)\,\KL(Q\Vert M_\beta)$.
3. $M_\beta(x)\ge\beta P(x)$ and $M_\beta(x)\ge(1-\beta)Q(x)$, so each log-ratio inside $D_\beta$ is bounded above: $D_\beta$ is **finite even when $P$ and $Q$ have disjoint support**.
4. As $\beta\to0$, $D_\beta/\beta\to\KL(P\Vert Q)$ (forward); as $\beta\to1$, $D_\beta/(1-\beta)\to\KL(Q\Vert P)$ (reverse).

**In words:** a **bounded** divergence whose $\beta$ slides its behavior between the two orders. It does **not** remove the choice: $\beta$ is a new knob.

- **Cost**: both **full** distributions at each position
- **Source**: Agarwal et al. (2024); their $\beta$ is a mixing weight, not the KL coefficient of RLLM-10, and which end is $\beta=0$ varies by paper
- **What we do not say**: which $\beta$, or which order, works best


In [ ]:
# Aside: the loop as pseudocode (the sampled-token version would replace `divergence` by the per-token teacher signal u above)
student = sft_checkpoint
teacher = frozen_model                    # never updated here

for step in range(max_steps):
    prompt = sample_prompt()
    completion = generate(student, prompt)         # on-policy sample

    # one distribution over the vocabulary per position, for each model
    p_student = student.next_token_dists(prompt, completion)
    p_teacher = teacher.next_token_dists(teacher_prompt, completion)   # no gradient
    # teacher_prompt is the prompt itself, or the prompt plus extra context (RLLM-12)

    loss = mean_over_positions(divergence(p_teacher, p_student))
    student = update(student, loss)               # no reward() anywhere


## Where the names come from

- **Distillation of a teacher's distribution**: Hinton, Vinyals and Dean (2015), *Distilling the Knowledge in a Neural Network*
- **Sequence-level distillation**: Kim and Rush (2016)
- **On-policy distillation for language models**: Agarwal et al. (2024), *On-Policy Distillation of Language Models: Learning from Self-Generated Mistakes*
- **Reverse KL with on-policy samples**: Gu et al. (2023), *MiniLLM*


## Reading for this session

- **Suggested**: for a paper, at least the **abstract and introduction**. For other items, the parts named
  - [Agarwal et al. (2024), *On-Policy Distillation of Language Models: Learning from Self-Generated Mistakes*](https://arxiv.org/abs/2306.13649): abstract and introduction: distilling on own samples
  - [Kevin Lu et al. (Thinking Machines Lab), *On-Policy Distillation*, blog post, Oct 2025](https://thinkingmachines.ai/blog/on-policy-distillation/): the "Implementation" and "Loss function: reverse KL" sections
- **Notes**
  - The blog's loss is reverse KL on the sampled token: this session's sampled-token variant, not a different method.
- **Additional** (4 more, for digging deeper), and the other sources this session cites: `reading_list.ipynb`
- Links were checked in October 2026; tell us if one has moved
